# Data Quarantine & Diagnostics
Use this notebook to confirm which features and target are active before modelling.
All paths are relative to the project root (`pemfc-gp/`).

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

def _find_project_root():
    start_dirs = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        start_dirs.append(os.path.dirname(os.path.realpath(nb)))
    start_dirs.append(os.getcwd())
    for start in start_dirs:
        d = os.path.abspath(start)
        for _ in range(10):
            if os.path.isdir(os.path.join(d, "src")) and os.path.isdir(os.path.join(d, "data")):
                return d
            d = os.path.dirname(d)
    raise RuntimeError("Cannot find pemfc-gp project root (expected src/ and data/)")

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from src.config import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    get_feature_bounds,
    get_feature_labels,
    get_target_label,
)
from src.data_parser import load_and_sanitize_data

X_train, y_train = load_and_sanitize_data("data/raw/", "data/processed/")
print(f"Loaded {len(y_train)} observations.")


In [ ]:
feature_labels = get_feature_labels()
target_label = get_target_label()
feature_to_index = {name: idx for idx, name in enumerate(FEATURE_COLUMNS)}
feature_stats = []
constant_features = []

print("=== ACTIVE CONFIGURATION ===")
print("Active features:", FEATURE_COLUMNS)
print("Active target:", TARGET_COLUMN)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}\n")

for idx, (feature_name, feature_label) in enumerate(zip(FEATURE_COLUMNS, feature_labels)):
    values = X_train[:, idx]
    unique_count = np.unique(values).size
    is_constant = np.isclose(values.min(), values.max())
    if is_constant:
        constant_features.append(feature_name)
    feature_stats.append({
        "name": feature_name,
        "label": feature_label,
        "min": float(values.min()),
        "max": float(values.max()),
        "mean": float(values.mean()),
        "unique_count": int(unique_count),
        "constant": bool(is_constant),
    })
    print(
        f"- {feature_label}: min={values.min():.4f}, max={values.max():.4f}, "
        f"mean={values.mean():.4f}, unique={unique_count}, constant={is_constant}"
    )

print(f"\n{target_label}: min={y_train.min():.4f}, max={y_train.max():.4f}, mean={y_train.mean():.4f}")

try:
    configured_bounds = get_feature_bounds()
    print("\nConfigured optimizer bounds:")
    for feature_name, feature_label, bound in zip(FEATURE_COLUMNS, feature_labels, configured_bounds):
        print(f"- {feature_label}: {bound}")
except KeyError as exc:
    configured_bounds = None
    print("\nOptimizer bounds are incomplete in src/config.py:")
    print(exc)

if constant_features:
    print("\nConstant features in the current training data:", constant_features)

if len(FEATURE_COLUMNS) > 2:
    print("\nPlot note: the quick plots below only show the first two active features.")


In [ ]:
print("=== BASIC DATA CHECKS ===")

if "TiO2_Loading" in feature_to_index:
    values = X_train[:, feature_to_index["TiO2_Loading"]]
    if np.any((values < 0.0) | (values > 10.0)):
        print("Warning: TiO2_Loading is outside the usual [0, 10] mg/cm^2 range.")
    else:
        print("TiO2_Loading stays within the usual [0, 10] mg/cm^2 range.")

if "RH" in feature_to_index:
    values = X_train[:, feature_to_index["RH"]]
    if np.any((values < 0.0) | (values > 100.0)):
        print("Warning: RH is outside the physical [0, 100]% range.")
    else:
        print("RH stays within the physical [0, 100]% range.")

if np.any(y_train < 0.0):
    print(f"Warning: {TARGET_COLUMN} contains negative values.")
else:
    print(f"{TARGET_COLUMN} contains no negative values.")


In [ ]:
if len(FEATURE_COLUMNS) < 2:
    print("Plot skipped: at least two active features are needed for the quick scatter plots.")
else:
    x_name, y_name = FEATURE_COLUMNS[:2]
    x_label, y_label = feature_labels[:2]
    x_values = X_train[:, 0]
    y_values = X_train[:, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    sc1 = ax1.scatter(x_values, y_train, c=y_values, cmap="coolwarm", edgecolors="k")
    ax1.set(title=f"{x_label} vs target", xlabel=x_label, ylabel=target_label)
    ax1.grid(True, ls="--", alpha=0.5)
    fig.colorbar(sc1, ax=ax1, label=y_label)

    sc2 = ax2.scatter(y_values, y_train, c=x_values, cmap="viridis", edgecolors="k")
    ax2.set(title=f"{y_label} vs target", xlabel=y_label, ylabel=target_label)
    ax2.grid(True, ls="--", alpha=0.5)
    fig.colorbar(sc2, ax=ax2, label=x_label)

    plt.tight_layout()
    plt.show()
